# ENERGICAL — Client Loyalty & Retention Model
### DataFest 2026 | The Outliers | June 2026

**Objective:** Predict client retention probability and generate personalized commercial strategies per wilaya segment.


## 0. Imports & Configuration

In [78]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

## 1 · Data Loading

In [81]:
transactions = pd.read_csv('data_clean.csv', parse_dates=['date_commande'])
products = pd.read_csv('energical_catalogue_produits.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data_clean.csv'

## 0. Imports & Configuration

###  Findings
- Transactions: 17,263 rows × 13 columns (2022–2024)
- Catalogue: 731 products × 10 columns
- Join key: `produit` (transactions) ↔ `nom_produit` (catalogue)

## 3. Feature engineering

### 2.1 · Merge transactions with catalogue

Left join to enrich transactions 

In [ ]:
transactions["produit"] = transactions["produit"].str.strip()
products["nom_produit"] = products["nom_produit"].str.strip()

df = transactions.merge(
    products,
    left_on="produit",
    right_on="nom_produit",
    how="left"
)

# Fill missing margin with 25% default
if "marge_estimee_pct" in df.columns:
    df["marge_estimee_pct"] = df["marge_estimee_pct"].fillna(25)
    df["profit_estime"] = (
        df["montant_da"] * 
        df["marge_estimee_pct"] / 100
    )
else:
    # If marge column doesn't exist, estimate 25%
    df["profit_estime"] = df["montant_da"] * 0.25
    print("⚠️ Using 25% margin estimate (marge_estimee_pct not in catalogue)")


⚠️ Using 25% margin estimate (marge_estimee_pct not in catalogue)


### 2.2 · Aggregate to client level

Each row in `customer_df` = 1 unique client. Features built:
- **Behavioral:** total orders, total spent, avg order, total quantity
- **Preference:** favorite category, favorite payment, most frequent wilaya
- **Economic:** avg margin, avg price, avg profit, avg restock delay

In [ ]:
customer_df = (
    df.groupby("id_client")
      .agg(
          returning=("nouveau_ou_fidele", "first"),

          wilaya=("wilaya",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          type_client=("type_client", "first"),

          favorite_category=("categorie_produit",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          favorite_payment=("moyen_paiement",
                  lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),

          total_orders=("id_transaction","count"),
          total_spent=("montant_da","sum"),
          average_order=("montant_da","mean"),
          total_quantity=("quantite","sum"),
          average_margin=("marge_estimee_pct","mean"),
          average_price=("prix_da","mean"),
          average_profit=("profit_estime","mean"),
          average_restock=("delai_reappro_jours","mean")
      )
      .reset_index()
)

KeyError: "Column(s) ['delai_reappro_jours', 'marge_estimee_pct', 'prix_da'] do not exist"

### 2.3  Filter for retention analysis + compute recency

Drop rows missing key fields, then compute `days_since_last_order` — the primary churn signal.

In [ ]:
df_retention = df.copy()

df_retention = df_retention.dropna(
    subset=[
        "produit",
        "quantite",
        "montant_da",
        "moyen_paiement"
    ]
)

In [ ]:
customer_df["average_quantity_per_order"] = (
    customer_df["total_quantity"] /
    customer_df["total_orders"]
)

df_retention["date_commande"] = pd.to_datetime(
    df_retention["date_commande"],
    dayfirst=True,
    errors="coerce"
)

reference_date = df_retention["date_commande"].max()

recency = (
    df_retention.groupby("id_client")["date_commande"]
    .max()
)

customer_df["days_since_last_order"] = (
    reference_date - customer_df["id_client"].map(recency)
).dt.days

## 3 · Target Variable & Preprocessing

### 3.1 · Define target variable and features

**Target:** `Returning` — 1 if the client is a returning customer, 0 if new.

**Features:**
- Categorical: `wilaya`, `type_client`, `favorite_category`, `favorite_payment`
- Numerical: `total_orders`, `total_spent`, `average_order`, `total_quantity`, `average_quantity_per_order`, `days_since_last_order`

In [ ]:
customer_df["Returning"] = (
    customer_df["returning"] == "Returning"
).astype(int)

features = [
    "wilaya",
    "type_client",
    "favorite_category",
    "favorite_payment",
    "total_orders",
    "total_spent",
    "average_order",
    "total_quantity",
    "average_quantity_per_order",
    "days_since_last_order"
]

X = customer_df[features]
y = customer_df["Returning"]

### 3.2 · Train / test split (80/20, stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### 3.3 · Preprocessing pipeline

- **Numerical:** median imputation
- **Categorical:** most-frequent imputation → one-hot encoding (`handle_unknown='ignore'`)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical = [
    "wilaya",
    "type_client",
    "favorite_category",
    "favorite_payment"
]

numerical = [
    "total_orders",
    "total_spent",
    "average_order",
    "total_quantity",
    "average_quantity_per_order",
    "days_since_last_order"
]

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical
    )
])

## 4 · Logistic Regression (Baseline)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

c:\Users\PCPRODZ\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['wilaya','type_client','favorite_category',...,'total_quantity', 'average_quantity_per_order','days_since_last_order']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specif

### 4.1 · Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, prob))

              precision    recall  f1-score   support

           0       0.88      0.99      0.93       710
           1       0.69      0.19      0.30       114

    accuracy                           0.88       824
   macro avg       0.79      0.59      0.62       824
weighted avg       0.86      0.88      0.84       824

[[700  10]
 [ 92  22]]
ROC AUC: 0.7353101062515444


In [ ]:
customer_df["Retention Score"] = model.predict_proba(X)[:, 1]

In [ ]:
def segment(score):
    if score >= 0.80:
        return "High"

    elif score >= 0.50:
        return "Medium"

    return "Low"

customer_df["Retention Level"] = customer_df["Retention Score"].apply(segment)

In [ ]:
def strategy(row):

    if row["Retention Score"] >= 0.80:
        return "Maintain loyalty through VIP rewards and exclusive offers."

    if row["days_since_last_order"] > 180:
        return "Launch a win-back campaign with personalized promotions."

    if row["total_orders"] <= 2:
        return "Offer a second-purchase discount to encourage repeat buying."

    if row["average_order"] < customer_df["average_order"].median():
        return "Promote bundles or complementary products to increase basket size."

    return "Send personalized recommendations based on the customer's preferred category."

customer_df["Recommended Strategy"] = customer_df.apply(strategy, axis=1)

## 5 · Random Forest (Final Model)

300 trees. Handles non-linearity, high-cardinality categoricals (48 wilayas), and class imbalance better than Logistic Regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['wilaya','type_client','favorite_category',...,'total_quantity', 'average_quantity_per_order','days_since_last_order']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specif

### 5.1 · Feature importances



In [ ]:
feature_names = rf_model.named_steps["prep"].get_feature_names_out()

In [ ]:
importances = rf_model.named_steps["clf"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

print(importance_df.head(20))

                                              Feature  Importance
1                                    num__total_spent    0.153232
5                          num__days_since_last_order    0.139709
0                                   num__total_orders    0.132484
2                                  num__average_order    0.126772
3                                 num__total_quantity    0.115681
4                     num__average_quantity_per_order    0.077985
8                                   cat__wilaya_Alger    0.015561
66                   cat__favorite_category_Sanitaire    0.013047
67                 cat__favorite_category_Électricité    0.012862
73      cat__favorite_payment_Paiement à la livraison    0.009696
69               cat__favorite_payment_CIB / EDAHABIA    0.009285
59                                cat__wilaya_Tlemcen    0.008709
47                         cat__wilaya_Sidi Bel Abbès    0.007648
41                                   cat__wilaya_Oran    0.007471
64        

###  Findings
- `total_spent`, `total_orders`, `average_order` = top 3 predictors — high-value clients return more
- `days_since_last_order` = 5th — recency matters but volume dominates
- Wilaya (Alger, Tlemcen, Oran) and category (Électricité, Sanitaire) have moderate weight
- **Business implication:** target clients with high historical spend but growing inactivity

## 7 · Export


In [ ]:
cols_to_export = [
    "id_client", "wilaya", "type_client",
    "favorite_category", "favorite_payment",
    "total_orders", "total_spent", "average_order",
    "total_quantity", "average_quantity_per_order",
    "days_since_last_order",
    "Retention Score", "Retention Level", "Recommended Strategy"
]

customer_df[cols_to_export].to_csv("customer_loyalty.csv", index=False)
print(f"✅ Exported customer_loyalty.csv — {len(customer_df)} clients")
print(customer_df["Retention Level"].value_counts())